In [ ]:
import numpy as np
import scipy.io as sio
import scipy.sparse as sp
from scipy.sparse.linalg import spsolve
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from scipy.ndimage import convolve
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd().parent / "data" / "hyperspectral_oil_spill"
device = "cuda" if torch.cuda.is_available() else "cpu"

train_files = [f for f in DATA_DIR.glob("*.mat") if f.stem not in ["GM01", "GM02"]]
test_files = [f for f in DATA_DIR.glob("*.mat") if f.stem in ["GM01", "GM02"]]

def evaluate_band_retention(train_files):
    water_vapor_bands = set(list(range(104, 114)) + list(range(148, 168)) + list(range(221, 224)))
    sample_img = sio.loadmat(train_files[0])["img"]
    total_raw_bands = sample_img.shape[2]
    valid_bands = [b for b in range(total_raw_bands) if b not in water_vapor_bands]
    mask = np.array([[1, -2,  1], [-2, 4, -2], [1, -2,  1]], dtype=float)
    all_scores = []
    
    for file_path in tqdm(train_files, desc="Evaluating Spectral Noise"):
        img = sio.loadmat(file_path)["img"]
        H, W, _ = img.shape
        scores = []
        for b in valid_bands:
            total_noise = 0.0
            for r in range(1, H - 1, 64):
                end = min(r + 64, H - 1)
                stripe = img[r-1:end+1, :, b].astype(float)
                total_noise += np.abs(convolve(stripe, mask)[1:-1, 1:-1]).sum()
            sigma_n = total_noise * np.sqrt(np.pi / 2) / (6 * (H - 2) * (W - 2))
            scores.append(sigma_n)
        all_scores.append(scores)
        
    avg_sigma = np.mean(all_scores, axis=0)
    clean_indices = np.argsort(avg_sigma)[:144]
    return np.sort([valid_bands[i] for i in clean_indices]).tolist()

CLEAN_BANDS = evaluate_band_retention(train_files)

In [ ]:
class CoTNetLayer(nn.Module):
    def __init__(self, dim, kernel_size=3):
        super().__init__()
        self.kernel_size = kernel_size
        self.key_embed = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.ReLU(inplace=True)
        )
        self.value_embed = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim)
        )
        factor = 4
        self.attention_embed = nn.Sequential(
            nn.Conv2d(2 * dim, 2 * dim // factor, 1, bias=False),
            nn.BatchNorm2d(2 * dim // factor),
            nn.ReLU(inplace=True),
            nn.Conv2d(2 * dim // factor, kernel_size * kernel_size * dim, 1)
        )

    def forward(self, x):
        bs, c, h, w = x.shape
        k1 = self.key_embed(x) 
        v = self.value_embed(x).view(bs, c, -1) 
        y = torch.cat([k1, x], dim=1) 
        att = self.attention_embed(y) 
        att = att.reshape(bs, c, self.kernel_size * self.kernel_size, h, w)
        att = att.mean(2, keepdim=False).view(bs, c, -1) 
        k2 = F.softmax(att, dim=-1) * v 
        k2 = k2.view(bs, c, h, w)
        return k1 + k2 

class True_SSTNet(nn.Module):
    def __init__(self, in_bands, embed_dim=128, num_classes=1):
        super().__init__()
        self.spectral_conv = nn.Sequential(
            nn.Conv3d(1, embed_dim, kernel_size=(7, 1, 1), padding=(3, 0, 0), bias=False),
            nn.BatchNorm3d(embed_dim),
            nn.ReLU(inplace=True),
            nn.Conv3d(embed_dim, embed_dim, kernel_size=(in_bands, 1, 1), bias=False),
            nn.BatchNorm3d(embed_dim),
            nn.ReLU(inplace=True)
        )
        self.cot1 = CoTNetLayer(dim=embed_dim)
        self.cot2 = CoTNetLayer(dim=embed_dim)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1) 
        x = self.spectral_conv(x) 
        x = x.squeeze(2) 
        x = self.cot1(x)
        x = self.cot2(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)

In [ ]:
from torchinfo import summary

model = True_SSTNet(in_bands=len(CLEAN_BANDS)).to(device)

summary(model, input_size=(1, len(CLEAN_BANDS), 11, 11), 
        col_names=["input_size", "output_size", "num_params", "mult_adds"])

In [ ]:
# Load the weights generated overnight
model.load_state_dict(torch.load("finetuned_sstnet.pth", map_location=device))
model.eval()
print("Successfully loaded fine-tuned weights!")

In [ ]:
def apply_erw_optimization(prob_map, img_rgb, beta=710, gamma=1e-5):
    """
    Optimizes the probability map using the ERW formulation from Paper 1.
    Builds a robust spatial graph without horizontal wrap-around edge artifacts.
    """
    H, W = prob_map.shape
    N = H * W
    P_init = prob_map.flatten()
    
    # Paper 1 recommends using the first Principal Component, but the grayscale 
    # intensity is a standard and fast approximation for structural boundaries.
    gray_img = np.mean(img_rgb, axis=2)
    
    # 1. Calculate Horizontal Edges (Right Neighbors)
    # diff_h shape: (H, W-1)
    diff_h = gray_img[:, :-1] - gray_img[:, 1:]
    weights_h = np.exp(-beta * (diff_h ** 2)).flatten()
    
    # 2. Calculate Vertical Edges (Bottom Neighbors)
    # diff_v shape: (H-1, W)
    diff_v = gray_img[:-1, :] - gray_img[1:, :]
    weights_v = np.exp(-beta * (diff_v ** 2)).flatten()
    
    # 3. Create absolute pixel indices
    idx = np.arange(N).reshape(H, W)
    
    # Horizontal edge coordinates (node_i, node_j)
    edges_h_i = idx[:, :-1].flatten()
    edges_h_j = idx[:, 1:].flatten()
    
    # Vertical edge coordinates (node_i, node_j)
    edges_v_i = idx[:-1, :].flatten()
    edges_v_j = idx[1:, :].flatten()
    
    # 4. Construct the symmetric Adjacency Matrix (W_adj)
    rows = np.concatenate((edges_h_i, edges_h_j, edges_v_i, edges_v_j))
    cols = np.concatenate((edges_h_j, edges_h_i, edges_v_j, edges_v_i))
    vals = np.concatenate((weights_h, weights_h, weights_v, weights_v))
    
    W_adj = sp.csr_matrix((vals, (rows, cols)), shape=(N, N))
    
    # 5. Construct the Laplacian (L = D - W_adj)
    D = sp.diags(W_adj.sum(axis=1).A1)
    L = D - W_adj
    
    # 6. Solve the ERW system: (L + gamma * I) * P_opt = gamma * P_init
    A = L + gamma * sp.eye(N, format='csr')
    b = gamma * P_init
    
    P_opt = spsolve(A, b)
    return P_opt.reshape((H, W))

def get_enhanced_rgb(img_array, rgb_bands=[29, 19, 9]):
    """
    Extracts and enhances RGB bands from the raw hyperspectral image.
    Red, Green, Blue bands (approx ~650nm, ~550nm, ~480nm in AVIRIS)
    """
    rgb_raw = img_array[:, :, rgb_bands].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    
    rgb_enhanced = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_enhanced[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
        
    return rgb_enhanced

def plot_full_scene(model, file_path, clean_bands, device, patch_size=11):
    mat = sio.loadmat(file_path)
    img = mat["img"]
    gt = mat["map"]
    
    # Extract RGB bands (approx ~650nm, ~550nm, ~480nm)
    rgb_raw = img[:, :, [29, 19, 9]].astype(np.float32)
    rgb_clean = np.where(rgb_raw < 0, 0, rgb_raw)
    rgb_img = np.zeros_like(rgb_clean)
    for c in range(3):
        channel = rgb_clean[:, :, c]
        p_low, p_high = np.percentile(channel[channel > 0], (2, 98))
        rgb_img[:, :, c] = np.clip((channel - p_low) / (p_high - p_low + 1e-8), 0, 1)
    
    # Process hyperspectral data
    img_clean = img[:, :, clean_bands].astype(np.float32)
    img_clean = (img_clean - np.min(img_clean)) / (np.max(img_clean) - np.min(img_clean) + 1e-8)
    pad = patch_size // 2
    img_padded = np.pad(img_clean, ((pad, pad), (pad, pad), (0, 0)), mode='symmetric')
    
    H, W = gt.shape
    raw_prob_map = np.zeros((H, W))
    valid_coords = [(r, c) for r in range(H) for c in range(W) if gt[r, c] >= 0]
    
    batch_size = 512 
    model.eval()
    
    with torch.inference_mode():
        for i in tqdm(range(0, len(valid_coords), batch_size), desc=f"Inferring {file_path.stem}"):
            batch_coords = valid_coords[i:i+batch_size]
            batch = [img_padded[r:r+patch_size, c:c+patch_size, :].transpose(2, 0, 1) for r, c in batch_coords]
            
            batch_tensor = torch.tensor(np.array(batch), dtype=torch.float32).to(device)
            probs = torch.sigmoid(model(batch_tensor)).cpu().numpy()
            
            # Populate the continuous probability map
            for (r, c), prob in zip(batch_coords, probs):
                raw_prob_map[r, c] = prob

    # Apply ERW spatial optimization
    print("Applying Extended Random Walker Optimization...")
    optimized_probs = apply_erw_optimization(raw_prob_map, rgb_img)
    
    # Calculate final binary predictions
    predictions = (optimized_probs > 0.5).astype(int)
    
    # Extract metrics using the valid coordinates
    all_probs = [optimized_probs[r, c] for r, c in valid_coords]
    all_targets = [gt[r, c] for r, c in valid_coords]
    preds_binary = (np.array(all_probs) > 0.5).astype(int)
    
    auc = roc_auc_score(all_targets, all_probs)
    precision = precision_score(all_targets, preds_binary, zero_division=0)
    recall = recall_score(all_targets, preds_binary, zero_division=0)
    f1 = f1_score(all_targets, preds_binary, zero_division=0)
    
    print(f"\n--- Full Scene Metrics for {file_path.stem} ---")
    print(f"AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n")

    # Visual Plotting
    fig, axs = plt.subplots(1, 3, figsize=(18, 8))
    axs[0].imshow(rgb_img)
    axs[0].set_title(f"False-Color RGB ({file_path.stem})")
    
    axs[1].imshow(gt == 1, cmap='magma')
    axs[1].set_title("Ground Truth Mask")
    
    axs[2].imshow(predictions, cmap='magma')
    axs[2].set_title("Model Prediction (ERW Optimized)")
    
    for ax in axs: 
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# Run the evaluation
plot_full_scene(model, test_files[0], CLEAN_BANDS, device)